# Topic Naming with OpenAI

Loads the BERTopic models and their topic info produced in `02-topic_model/`,
sends representative documents per cluster to GPT to obtain:

1. A **description** of each topic cluster.
2. An **enhanced (synthesised) description**.
3. A short **cluster name**.

Prompts are loaded from `prompts.yaml`. Results are saved as TSV files alongside
the topic models in `assets/topic_models/`.

In [1]:
import os
import yaml
import numpy as np
import pandas as pd
from openai import OpenAI

In [2]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
ASSETS_DIR     = os.path.join('..', 'assets')
MODELS_DIR     = os.path.join(ASSETS_DIR, 'topic_models')
EMBEDDINGS_DIR = os.path.join(ASSETS_DIR, 'embeddings')

# Number of representative documents to send per cluster
TOP_N_DOCS = 5

# OpenAI model
MODEL = 'gpt-4.1-nano'

# ── Load prompts ──────────────────────────────────────────────────────────────
with open('prompts.yaml') as f:
    prompts = yaml.safe_load(f)

THEME = prompts['theme']
THEME_DESCRIPTION = prompts['theme_description']

print(f'Theme: {THEME}')
print(f'Description: {THEME_DESCRIPTION[:80]}...')

Theme: Synthetic Biology
Description: Synthetic biology is an interdisciplinary field that combines principles from bi...


In [3]:
# ── OpenAI client ─────────────────────────────────────────────────────────────
api_key = open('openai.key').read().strip()
client = OpenAI(api_key=api_key)

## 1. Load topic models and corpora

In [4]:
# Topic-level info (from BERTopic's get_topic_info())
teams_info  = pd.read_csv(os.path.join(MODELS_DIR, 'teams_topic_info.txt'),  sep='\t')
papers_info = pd.read_csv(os.path.join(MODELS_DIR, 'papers_topic_info.txt'), sep='\t')

# Document-level topic assignments
teams_doc_topics  = pd.read_csv(os.path.join(MODELS_DIR, 'teams_doc_topics.txt'),  sep='\t')
papers_doc_topics = pd.read_csv(os.path.join(MODELS_DIR, 'papers_doc_topics.txt'), sep='\t')

# Full corpora (with text)
teams_corpus  = pd.read_csv(os.path.join(EMBEDDINGS_DIR, 'teams_corpus.txt'),  sep='\t')
papers_corpus = pd.read_csv(os.path.join(EMBEDDINGS_DIR, 'papers_corpus.txt'), sep='\t')

# Merge text into doc-topics
teams_df  = teams_doc_topics.merge(teams_corpus,  on='UT', how='left')
papers_df = papers_doc_topics.merge(papers_corpus, on='id', how='left')

print(f'Teams:  {len(teams_df):,} docs, {teams_info.Topic.max() + 1} topics')
print(f'Papers: {len(papers_df):,} docs, {papers_info.Topic.max() + 1} topics')

Teams:  4,548 docs, 161 topics
Papers: 24,202 docs, 263 topics


## 2. Helper functions

In [5]:
def ask_gpt(
    system_prompt: str,
    user_prompt: str,
    model: str = MODEL,
    temperature: float = 0.1,
    max_tokens: int = 500,
) -> str:
    """Send a system + user prompt to OpenAI and return the assistant reply."""
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': user_prompt},
        ],
    )
    return response.choices[0].message.content


def get_representative_texts(df: pd.DataFrame, topic_id: int, top_n: int = TOP_N_DOCS) -> str:
    """Return the top-N documents of a topic joined with ##### separators.

    The text is truncated to ~14 000 chars (~3 500 tokens) to stay within
    context limits.
    """
    subset = df[df['topic'] == topic_id]
    texts = subset['text'].head(top_n).tolist()
    bulk = ' ##### '.join(texts)
    return bulk[:14_000]


def fmt_prompt(template: str, **kwargs) -> str:
    """Format a prompt template with the given keyword arguments."""
    result = template
    for key, value in kwargs.items():
        result = result.replace('{' + key + '}', str(value))
    return result

In [6]:
def name_topics(
    df: pd.DataFrame,
    topic_info: pd.DataFrame,
) -> pd.DataFrame:
    """For each non-outlier topic, get description, enhanced description, and name.

    Returns a DataFrame with columns: topic, description, enhanced_description, name.
    """
    p_desc = prompts['cluster_description']
    p_enh  = prompts['cluster_description_enhanced']
    p_name = prompts['cluster_name']

    rows = []
    topic_ids = sorted(topic_info[topic_info['Topic'] != -1]['Topic'].unique())

    for tid in topic_ids:
        print(f'  Topic {tid} ...', end=' ', flush=True)
        cluster_text = get_representative_texts(df, tid)

        # Step 1: cluster description
        description = ask_gpt(
            system_prompt=fmt_prompt(p_desc['system'], theme=THEME, theme_description=THEME_DESCRIPTION),
            user_prompt=fmt_prompt(p_desc['user'], cluster_text=cluster_text),
            temperature=0.2,
        )

        # Step 2: enhanced (synthesised) description
        enhanced = ask_gpt(
            system_prompt=fmt_prompt(p_enh['system'], theme=THEME),
            user_prompt=fmt_prompt(p_enh['user'], cluster_description=description),
            temperature=0.1,
        )

        # Step 3: short name
        name = ask_gpt(
            system_prompt=fmt_prompt(p_name['system'], theme=THEME, theme_description=THEME_DESCRIPTION),
            user_prompt=fmt_prompt(p_name['user'], cluster_description=description),
            max_tokens=60,
            temperature=0.3,
        )
        name = name.strip().strip('"').strip("'")

        print(name)
        rows.append({
            'topic': tid,
            'name': name,
            'description': enhanced,
            'raw_description': description,
        })

    return pd.DataFrame(rows)

## 3. Name iGEM Teams topics

In [7]:
print('Naming iGEM Teams topics...')
teams_names = name_topics(teams_df, teams_info)
teams_names

Naming iGEM Teams topics...
  Topic 0 ... Synthetic Biology-Enabled Disease Diagnostics and Biosensing Technologies
  Topic 1 ... Light-Inducible Systems
  Topic 2 ... Synthetic Biology for Microbial Detection and Biofilm Control
  Topic 3 ... Synthetic Biology for PET Plastic Degradation and Recycling
  Topic 4 ... Synthetic Biology for Cancer and Disease Therapeutics
  Topic 5 ... Microbial Diagnostic and Therapeutic Platforms
  Topic 6 ... Synthetic Biology for Diagnostic and Therapeutic Biosystems
  Topic 7 ... Synthetic Biology for Genetic Control and Diagnostics
  Topic 8 ... Synthetic Biology Design and Automation
  Topic 9 ... Synthetic Biology Tools for Genetic and Molecular Control
  Topic 10 ... Synthetic Biology for Sustainable Environmental and Resource Solutions
  Topic 11 ... Synthetic Biology Diagnostics
  Topic 12 ... Synthetic Biology for Environmental and Health Monitoring
  Topic 13 ... Bioproduction and Cellular Engineering
  Topic 14 ... Bioactive Compound Synthes

,topic,name,description,raw_description
0,0,Synthetic Biology-Enabled Disease Diagnostics ...,This cluster focuses on the advancement of inn...,The common topic of these texts is the develop...
1,1,Light-Inducible Systems,This cluster centers on the development and ap...,The main topic of this cluster is **Light-Indu...
2,2,Synthetic Biology for Microbial Detection and ...,This cluster focuses on harnessing synthetic b...,The main topic of this cluster is the applicat...
3,3,Synthetic Biology for PET Plastic Degradation ...,This cluster focuses on leveraging synthetic b...,The common topic across these texts is the app...
4,4,Synthetic Biology for Cancer and Disease Thera...,This cluster centers on leveraging synthetic b...,The common topic of this cluster is **Syntheti...
...,...,...,...,...
156,156,Microbial Genetic Parts and Tools,This cluster focuses on advancing the toolkit ...,The main topic of this cluster is the developm...
157,157,Environmental Biosensing,This collection of works centers on harnessing...,The common topic across these texts is the app...
158,158,Synthetic Biology for Eye Disease Prevention a...,The collective focus of these works centers on...,The common topic of these texts is the applica...
159,159,CRISPR Diagnostic Technologies,The collective focus of these texts centers on...,The common topic of these texts is the applica...


## 4. Name SynBio Papers topics

In [8]:
print('Naming SynBio Papers topics...')
papers_names = name_topics(papers_df, papers_info)
papers_names

Naming SynBio Papers topics...
  Topic 0 ... Genetic Circuit Engineering
  Topic 1 ... Synthetic Nucleic Acid Engineering
  Topic 2 ... Plant Biosynthetic Pathways
  Topic 3 ... Synthetic CpG ODNs in Immunomodulation
  Topic 4 ... Ethics and Governance of Synthetic Biology
  Topic 5 ... Genetic Circuits and Logic Devices
  Topic 6 ... Oligonucleotide Modification and Immobilization
  Topic 7 ... Synthetic Genome Engineering
  Topic 8 ... Yeast Gene Regulation Cluster
  Topic 9 ... Synthetic Biological Systems and Circuits
  Topic 10 ... Synthetic Biology-Driven Biomaterials
  Topic 11 ... DNA Repair and Epigenetics
  Topic 12 ... Synthetic Biology Design and Modeling
  Topic 13 ... RNA Regulatory Systems
  Topic 14 ... Synthetic Biology in Cancer Immunotherapy
  Topic 15 ... Synthetic Protein Engineering
  Topic 16 ... DNA Nanotechnology
  Topic 17 ... Synthetic Biology in Gene Cloning and Protein Expression
  Topic 18 ... Synthetic Biology in Biomedical Engineering
  Topic 19 ... Plan

,topic,name,description,raw_description
0,0,Genetic Circuit Engineering,This cluster centers on the engineering and re...,The common topic of this cluster is the design...
1,1,Synthetic Nucleic Acid Engineering,This cluster centers on the application of syn...,The common topic of these texts is the applica...
2,2,Plant Biosynthetic Pathways,This collection of texts focuses on leveraging...,The common topic of these texts is the applica...
3,3,Synthetic CpG ODNs in Immunomodulation,This cluster centers on the immunostimulatory ...,The common topic of this cluster is the immuno...
4,4,Ethics and Governance of Synthetic Biology,This cluster focuses on the multifaceted ethic...,"The common topic of this cluster is **Ethical,..."
...,...,...,...,...
258,258,Chloroplast and Plastid Genome Engineering,The collective focus of these texts centers on...,The common topic across these texts is **Synth...
259,259,CRISPR Yeast Synthetic Biology,This cluster centers on the utilization and en...,The main topic of this cluster is the applicat...
260,260,Computational Synthetic Biology,This cluster centers on the development and ut...,The common topic across these texts is the dev...
261,261,DNA Repair Systems in Synthetic Biology,The cluster focuses on the integral role of Pr...,The main topic of this cluster is the role of ...


## 5. Save results

In [9]:
teams_names.to_csv(os.path.join(MODELS_DIR, 'teams_topic_names.txt'), sep='\t', index=False)
papers_names.to_csv(os.path.join(MODELS_DIR, 'papers_topic_names.txt'), sep='\t', index=False)

print(f'Saved to {MODELS_DIR}')
for f in sorted(os.listdir(MODELS_DIR)):
    print(f'  {f}')

Saved to ../assets/topic_models
  papers_doc_topics.txt
  papers_grid_search.txt
  papers_topic_info.txt
  papers_topic_model
  papers_topic_names.txt
  teams_doc_topics.txt
  teams_grid_search.txt
  teams_topic_info.txt
  teams_topic_model
  teams_topic_names.txt
